# Notebook 4 — GPT-style Dataset + Minimal Sequence Model

From here on, **the structure of the target itself** changes.

Previously `y` was a single next character; now `y` is a sequence too.

- `x = [t1, t2, ..., tT]`
- `y = [t2, t3, ..., t(T+1)]`

In [ ]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("shakespeare.txt").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", "shakespeare.txt")
text = open("shakespeare.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

In [ ]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor

## 1. GPT-style dataset

In [ ]:
class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

xb, yb = next(iter(loader))
print("xb.shape:", xb.shape)
print("yb.shape:", yb.shape)

xb.shape: torch.Size([64, 32])
yb.shape: torch.Size([64, 32])


In [ ]:
xb[0]

tensor([50, 50,  1, 14, 53, 46, 43, 51, 47, 39, 10,  1, 47, 44,  1, 63, 53, 59,
         1, 46, 39, 42,  0, 40, 59, 58,  1, 50, 53, 53, 49, 43])

## 2. A minimal sequence model without attention

In [ ]:
class TinySequenceLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok = self.token_embedding(x)            # (B, T, C)
        pos = self.position_embedding(pos)[None] # (1, T, C)
        h = tok + pos
        logits = self.lm_head(h)                 # (B, T, V)
        return logits

model = TinySequenceLM(vocab_size, block_size)
logits = model(xb)
print("logits.shape:", logits.shape)

logits.shape: torch.Size([64, 32, 65])


## 3. Loss

In [ ]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

print("initial loss:", sequence_cross_entropy(logits, yb).item())

initial loss: 4.565671443939209


## 4. Training

In [ ]:
def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TinySequenceLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

epoch  0 | train loss 3.0562
epoch  1 | train loss 2.5599
epoch  2 | train loss 2.5059
epoch  3 | train loss 2.4900
epoch  4 | train loss 2.4806
epoch  5 | train loss 2.4751
epoch  6 | train loss 2.4714
epoch  7 | train loss 2.4689
epoch  8 | train loss 2.4669
epoch  9 | train loss 2.4637
epoch 10 | train loss 2.4640
epoch 11 | train loss 2.4618
epoch 12 | train loss 2.4647
epoch 13 | train loss 2.4634
epoch 14 | train loss 2.4634
epoch 15 | train loss 2.4595
epoch 16 | train loss 2.4591
epoch 17 | train loss 2.4600
epoch 18 | train loss 2.4603
epoch 19 | train loss 2.4582
epoch 20 | train loss 2.4621
epoch 21 | train loss 2.4564
epoch 22 | train loss 2.4594
epoch 23 | train loss 2.4590
epoch 24 | train loss 2.4563
epoch 25 | train loss 2.4603
epoch 26 | train loss 2.4548
epoch 27 | train loss 2.4547
epoch 28 | train loss 2.4573
epoch 29 | train loss 2.4566
epoch 30 | train loss 2.4579
epoch 31 | train loss 2.4578
epoch 32 | train loss 2.4589
epoch 33 | train loss 2.4579
epoch 34 | tra

## 5. Sampling

In [ ]:
@torch.no_grad()
def sample_sequence_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample_sequence_model(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=400))

ROMEO:
I itrt wrayourandolif-me HAsindst courer ch ffeefath o n t he,
Ungs thinel word bbedy mad ppimed, feanck faves d ing, rfit osherithef ishangovey m the;

Ped w s'ld fownd th ad y.
Tithever br:


HARYo wine.

Thowout thrr'soushe ndon; JUKETI anes oud t's kithity ouryeayoncobuspu haiming powho are leldorure: monineme lvet wll.
y; ullu anf hatothueru ce 'e'd:
Yoran t t hals s, thert stoowhisttasesora


## 6. Summary

- The target is now a sequence as well.
- The output shape is `(B, T, V)`.
- Positional embedding is introduced for the first time.
- There is no attention yet, but the data and output interface of a GPT is already in place.